# NOTEBOOK #02 — Calidad y Limpieza

**Proyecto Integrador — Minería de Datos I — Tec. Sup. Ciencia de Datos — ITSE**  
**Integrantes:** Daniela Fenandez — Julio Nahuel Gomez  
**Profesor:** Fernando Elias Mubarqui  
**Año:** 2026

**Dataset:** Usuarios de plataforma de streaming  
**Objetivo:** Tratar los problemas identificados en la inspección inicial. Cada decisión de limpieza se justifica con la evidencia observada, se documenta la acción aplicada y el impacto en el dataset.

In [21]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_json('../data/raw/streaming_users_dirty.csv')

print(f"Dataset cargado: {df.shape[0]} filas, {df.shape[1]} columnas")

Dataset cargado: 8160 filas, 8 columnas


In [22]:
#Creamos una copia llamada `df_clean` sobre la que vamos a hacer todas las transformaciones. 
# El `df` original queda intacto. También armamos la función `auditar()` para registrar el impacto de cada paso.

df_clean = df.copy()

print("Copia creada correctamente.")
print(f"Filas iniciales: {df_clean.shape[0]}")

Copia creada correctamente.
Filas iniciales: 8160


In [23]:
log = []

def auditar(paso, descripcion, d):
    log.append({
        'Paso': paso,
        'Descripción': descripcion,
        'Filas': d.shape[0],
        'Nulos': int(d.isnull().sum().sum()),
        'Retención (%)': round(d.shape[0] / df.shape[0] * 100, 2)
    })

auditar(0, 'Dataset original cargado', df_clean)

## EVIDENCIA DE VARIABLES CATEGÓRICAS

Antes de armar los diccionarios de mapeo, listamos todas las categorías detectadas tal cual aparecen en el dato crudo. Esto deja registrada la evidencia exacta sobre la que se decide cada mapeo.

In [24]:
for col in ['subscription_plan', 'country', 'favorite_genre']:
    valores = sorted(str(v) for v in df_clean[col].unique())
    print(f"\nColumna '{col}': {len(valores)} categorías detectadas")
    print("-" * 60)
    print(valores)


Columna 'subscription_plan': 15 categorías detectadas
------------------------------------------------------------
['BASICO', 'Basic', 'Básico', 'Estándar', 'Estándar ', 'PREMIUM', 'Premium', 'Premium ', 'Premiun', 'STANDARD', 'Std', 'basico', 'básico', 'estandar', 'premium']

Columna 'country': 26 categorías detectadas
------------------------------------------------------------
['ARG', 'Argentina', 'Argentina ', 'BRA', 'Brasil', 'Brazil', 'CHL', 'COL', 'Chile', 'Chile ', 'Colombia', 'MEX', 'Mexico', 'México', 'PER', 'Peru', 'Perú', 'URY', 'Uruguay', 'argentina', 'brasil', 'chile', 'colombia', 'méxico', 'perú', 'uruguay']

Columna 'favorite_genre': 29 categorías detectadas
------------------------------------------------------------
['ACCIÓN', 'Acción', 'Action', 'COMEDIA', 'CRIME', 'Comedia', 'Comedia ', 'Crime', 'Crimen', 'DOC', 'DRAMA', 'Documental', 'Documentary', 'Drama', 'Drama ', 'ROMANCE', 'Romance', 'Romance ', 'THRILLER', 'Thriller', 'Thriller ', 'accion', 'comedy', 'crime'

## ELIMINACIÓN DE DUPLICADOS

**Evidencia:** En la inspección inicial detectamos 126 registros duplicados, es decir, filas exactamente iguales en todas sus columnas.  
**Acción:** Se eliminan los duplicados conservando la primera ocurrencia de cada registro.  
**Justificación:** Un registro duplicado no aporta información nueva y puede distorsionar los análisis estadísticos.

In [25]:
antes = df_clean.shape[0]
df_clean = df_clean.drop_duplicates()

print(f"Filas antes: {antes}")
print(f"Filas después: {df_clean.shape[0]}")
print(f"Duplicados eliminados: {antes - df_clean.shape[0]}")

auditar(1, 'Eliminación de duplicados', df_clean)

Filas antes: 8160
Filas después: 8034
Duplicados eliminados: 126


## NORMALIZACIÓN DE VARIABLES CATEGÓRICAS

### subscription_plan

**Evidencia:** Se detectaron 15 variantes para 3 planes: mayúsculas, minúsculas, con/sin tilde, abreviaciones (Std, Basic, Premiun).  
**Acción:** Se unifican todas las variantes en tres valores estándar: Básico, Estándar, Premium.  
**Justificación:** Sin esta normalización, los análisis agrupados por plan darían resultados incorrectos.

In [26]:
df_clean['subscription_plan'] = df_clean['subscription_plan'].str.strip().str.lower()

plan_map = {
    'básico': 'Básico', 'basico': 'Básico', 'basic': 'Básico',
    'estándar': 'Estándar', 'estandar': 'Estándar', 'std': 'Estándar', 'standard': 'Estándar',
    'premium': 'Premium', 'premiun': 'Premium'
}

df_clean['subscription_plan'] = df_clean['subscription_plan'].map(plan_map)

print("Valores únicos después de normalizar:")
print(df_clean['subscription_plan'].value_counts())

auditar(2, 'Normalización subscription_plan', df_clean)

Valores únicos después de normalizar:
subscription_plan
Básico      3609
Estándar    2833
Premium     1592
Name: count, dtype: int64


### country

**Evidencia:** Se detectaron 26 variantes para 7 países: nombres completos, códigos de 3 letras (MEX, ARG, BRA), minúsculas y espacios extra.  
**Acción:** Se unifican todas las variantes al nombre completo del país en español.  
**Justificación:** Sin esta normalización, un mismo país aparecería como categorías distintas en los análisis.

In [27]:
df_clean['country'] = df_clean['country'].str.strip().str.lower()

country_map = {
    'brasil': 'Brasil', 'brazil': 'Brasil', 'bra': 'Brasil',
    'colombia': 'Colombia', 'col': 'Colombia',
    'uruguay': 'Uruguay', 'ury': 'Uruguay',
    'perú': 'Perú', 'peru': 'Perú', 'per': 'Perú',
    'chile': 'Chile', 'chl': 'Chile',
    'argentina': 'Argentina', 'arg': 'Argentina',
    'méxico': 'México', 'mexico': 'México', 'mex': 'México'
}

df_clean['country'] = df_clean['country'].map(country_map)

print("Valores únicos después de normalizar:")
print(df_clean['country'].value_counts())

auditar(3, 'Normalización country', df_clean)

Valores únicos después de normalizar:
country
Chile        1167
Brasil       1164
México       1162
Uruguay      1146
Colombia     1145
Perú         1139
Argentina    1111
Name: count, dtype: int64


### favorite_genre

**Evidencia:** Se detectaron múltiples variantes para cada género: mezcla de español e inglés, mayúsculas, minúsculas, errores tipográficos (thriler, accion) y abreviaciones (DOC).  
**Acción:** Se unifican todas las variantes al nombre en español.  
**Justificación:** Sin esta normalización, un mismo género aparecería como categorías distintas en los análisis.

In [28]:
df_clean['favorite_genre'] = df_clean['favorite_genre'].str.strip().str.lower()

genre_map = {
    'crime': 'Crimen', 'crimen': 'Crimen',
    'thriller': 'Thriller', 'thriler': 'Thriller',
    'drama': 'Drama',
    'acción': 'Acción', 'accion': 'Acción', 'action': 'Acción',
    'romance': 'Romance',
    'comedia': 'Comedia', 'comedy': 'Comedia',
    'documental': 'Documental', 'documentary': 'Documental', 'doc': 'Documental'
}

df_clean['favorite_genre'] = df_clean['favorite_genre'].map(genre_map)

print("Valores únicos después de normalizar:")
print(df_clean['favorite_genre'].value_counts(dropna=False))

auditar(4, 'Normalización favorite_genre', df_clean)

Valores únicos después de normalizar:
favorite_genre
Comedia       1141
Drama         1121
Romance       1113
Documental    1111
Acción        1110
Thriller      1109
Crimen        1089
NaN            240
Name: count, dtype: int64


## TRATAMIENTO DE VALORES FALTANTES

### favorite_genre (240 nulos)

**Evidencia:** Se detectaron 240 registros sin género favorito.  
**Acción:** Se imputan con la moda (valor más frecuente).  
**Justificación:** Al ser una variable categórica sin orden, la moda es la medida más adecuada para imputar. Eliminar 240 filas representaría perder el 3% del dataset.

In [29]:
moda_genre = df_clean['favorite_genre'].mode()[0]
print(f"Moda de favorite_genre: {moda_genre}")

df_clean['favorite_genre'] = df_clean['favorite_genre'].fillna(moda_genre)

print(f"Nulos restantes en favorite_genre: {df_clean['favorite_genre'].isnull().sum()}")

auditar(5, 'Imputación favorite_genre con moda', df_clean)

Moda de favorite_genre: Comedia
Nulos restantes en favorite_genre: 0


### last_login_date (320 nulos + formato mixto)

**Evidencia:** Se detectaron 320 registros sin fecha de último login. Además, la columna tiene fechas escritas en distintos formatos mezclados en la misma columna.  
**Acción:** Se eliminan los 320 registros sin fecha. Luego se convierte la columna usando `format='mixed'` para que pandas interprete cada fecha con su propio formato. Las fechas que siguen sin poder convertirse (valores centinela como `0000-00-00` o fechas imposibles como `31-02-2022`) se eliminan.  
**Justificación:** Con `format='mixed'` evitamos perder fechas válidas escritas en otro formato. Solo se eliminan las realmente inválidas.

In [30]:
antes = df_clean.shape[0]
df_clean = df_clean.dropna(subset=['last_login_date'])
print(f"Filas después de eliminar nulos: {df_clean.shape[0]} (eliminadas: {antes - df_clean.shape[0]})")

auditar(6, 'Eliminación nulos last_login_date', df_clean)

antes = df_clean.shape[0]
df_clean['last_login_date'] = pd.to_datetime(df_clean['last_login_date'], errors='coerce', format='mixed')
df_clean = df_clean.dropna(subset=['last_login_date'])

print(f"Filas después de parsear fechas: {df_clean.shape[0]} (eliminadas: {antes - df_clean.shape[0]})")
print(f"Tipo de dato de last_login_date: {df_clean['last_login_date'].dtype}")

auditar(7, "Parseo de fechas (format='mixed') y eliminación de fechas inválidas", df_clean)

Filas después de eliminar nulos: 7714 (eliminadas: 320)
Filas después de parsear fechas: 7650 (eliminadas: 64)
Tipo de dato de last_login_date: datetime64[us]


## TRATAMIENTO DE VALORES IMPOSIBLES EN VARIABLES NUMÉRICAS

Separamos los valores que directamente no pueden existir en la realidad. Esto no es tratamiento de outliers: es dato inválido. Lo sacamos primero para que los estadísticos que calculemos después no queden contaminados.

### age

**Evidencia:** Se detectaron valores de edad menores a 0 y mayores a 100. Una edad negativa o mayor a 100 es imposible para un usuario de streaming.  
**Acción:** Se eliminan los registros con edad fuera del rango válido (0–100).  
**Justificación:** No es posible estimar la edad correcta de un registro con valor inválido.

In [31]:
antes = df_clean.shape[0]
df_clean = df_clean[(df_clean['age'] >= 0) & (df_clean['age'] <= 100)]
despues = df_clean.shape[0]

print(f"Filas eliminadas por edad inválida: {antes - despues}")
print(f"Filas restantes: {despues}")
print(f"Rango de edad válido: {df_clean['age'].min()} - {df_clean['age'].max()}")

auditar(8, 'Filtrado de age imposible (0-100)', df_clean)

Filas eliminadas por edad inválida: 70
Filas restantes: 7580
Rango de edad válido: 0 - 80


### monthly_watch_time_mins (valores imposibles)

**Evidencia:** Se detectaron valores negativos (mínimo -120) y valores por encima del máximo físicamente posible en un mes: 44.640 minutos (31 días × 24 horas × 60 minutos). Valores como 99999 superan ese límite y no pueden ser un tiempo de visualización real.  
**Acción:** Se eliminan los registros negativos y los que superan los 44.640 minutos.  
**Justificación:** Estos valores no tienen ninguna interpretación válida, a diferencia de los outliers estadísticos que tratamos más abajo.

In [32]:
antes = df_clean.shape[0]
MAX_FISICO_MENSUAL = 31 * 24 * 60

df_clean = df_clean[(df_clean['monthly_watch_time_mins'] >= 0) | (df_clean['monthly_watch_time_mins'].isnull())]
df_clean = df_clean[(df_clean['monthly_watch_time_mins'] <= MAX_FISICO_MENSUAL) | (df_clean['monthly_watch_time_mins'].isnull())]

print(f"Filas eliminadas por valor imposible: {antes - df_clean.shape[0]}")
print(f"Filas restantes: {df_clean.shape[0]}")

auditar(9, 'Eliminación monthly_watch_time_mins imposible (negativos o >44.640)', df_clean)

Filas eliminadas por valor imposible: 75
Filas restantes: 7505


### monthly_watch_time_mins (193 nulos)

**Evidencia:** Se detectaron 193 registros sin tiempo de visualización mensual.  
**Acción:** Se imputan con la mediana, calculada ya sin los valores imposibles del paso anterior.  
**Justificación:** Al ser una variable numérica continua con cola de valores altos, la mediana es más robusta que la media para imputar.

In [33]:
mediana_watch = df_clean['monthly_watch_time_mins'].median()
print(f"Mediana de monthly_watch_time_mins: {mediana_watch}")

df_clean['monthly_watch_time_mins'] = df_clean['monthly_watch_time_mins'].fillna(mediana_watch)

print(f"Nulos restantes: {df_clean['monthly_watch_time_mins'].isnull().sum()}")

auditar(10, 'Imputación monthly_watch_time_mins con mediana', df_clean)

Mediana de monthly_watch_time_mins: 758.7
Nulos restantes: 0


## TRATAMIENTO DE OUTLIERS ESTADÍSTICOS

Estos valores sí son físicamente posibles, solo que están lejos del resto de los datos. Calculamos los límites con IQR usando k=1.5 y k=3.0.

### monthly_watch_time_mins

In [34]:
Q1 = df_clean['monthly_watch_time_mins'].quantile(0.25)
Q3 = df_clean['monthly_watch_time_mins'].quantile(0.75)
IQR = Q3 - Q1
limite_15 = Q3 + 1.5 * IQR
limite_30 = Q3 + 3.0 * IQR

print(f"Q1={Q1:.1f}  Q3={Q3:.1f}  IQR={IQR:.1f}")
print(f"Límite k=1.5: {limite_15:.1f} -> {(df_clean['monthly_watch_time_mins'] > limite_15).sum()} valores por encima")
print(f"Límite k=3.0: {limite_30:.1f} -> {(df_clean['monthly_watch_time_mins'] > limite_30).sum()} valores por encima")

media_actual = df_clean['monthly_watch_time_mins'].mean()
mediana_actual = df_clean['monthly_watch_time_mins'].median()
media_sin = df_clean[df_clean['monthly_watch_time_mins'] <= limite_15]['monthly_watch_time_mins'].mean()
mediana_sin = df_clean[df_clean['monthly_watch_time_mins'] <= limite_15]['monthly_watch_time_mins'].median()

print(f"\nMedia actual: {media_actual:.1f}   Mediana actual: {mediana_actual:.1f}")
print(f"Media sin outliers k=1.5: {media_sin:.1f}   Mediana sin outliers: {mediana_sin:.1f}")
print(f"\nAsimetría actual: {df_clean['monthly_watch_time_mins'].skew():.3f}")

Q1=500.1  Q3=1034.8  IQR=534.7
Límite k=1.5: 1836.8 -> 120 valores por encima
Límite k=3.0: 2638.9 -> 108 valores por encima

Media actual: 796.2   Mediana actual: 758.7
Media sin outliers k=1.5: 754.3   Mediana sin outliers: 758.7

Asimetría actual: 2.512


**Interpretación:** la mediana prácticamente no se mueve al sacar los outliers, pero la asimetría es alta, lo que confirma una cola larga hacia la derecha. Los valores más altos corresponden a usuarios que ven mucho contenido, no a errores de carga.

**Decisión:** no eliminamos estas filas. Winsorizamos: acotamos el valor al límite de k=1.5 sin tocar la fila. Así conservamos el registro completo y corregimos la distorsión que esos valores generan en la asimetría.

In [35]:
limite_winsor = limite_15
df_clean['monthly_watch_time_mins'] = df_clean['monthly_watch_time_mins'].clip(upper=limite_winsor)

print("Filas eliminadas: 0 (no se elimina ninguna fila, solo se acota el valor)")
print(f"Filas restantes: {df_clean.shape[0]}")
print(f"Asimetría después de winsorizar: {df_clean['monthly_watch_time_mins'].skew():.3f}")
print(f"Rango final: {df_clean['monthly_watch_time_mins'].min()} - {df_clean['monthly_watch_time_mins'].max()}")

auditar(11, f'Winsorización monthly_watch_time_mins (k=1.5)', df_clean)

Filas eliminadas: 0 (no se elimina ninguna fila, solo se acota el valor)
Filas restantes: 7505
Asimetría después de winsorizar: 0.254
Rango final: 0.0 - 1836.85


### customer_support_tickets

**Evidencia:** La distribución muestra valores naturales (0 a 5 tickets), pero hay un salto directo a 99 y 150 sin ningún valor intermedio. Ese salto es la firma típica de un valor centinela, no de usuarios con muchos tickets reales.

In [36]:
print("Distribución de customer_support_tickets:")
print(df_clean['customer_support_tickets'].value_counts().sort_index())

Distribución de customer_support_tickets:
customer_support_tickets
-1        25
 0      3479
 1      2750
 2       899
 3       213
 4        61
 5        12
 99       34
 150      32
Name: count, dtype: int64


**Decisión:** tratamos -1, 99 y 150 como valores inválidos (no como outliers estadísticos) y eliminamos esas filas. El resto de la variable (0 a 5 tickets) queda sin tocar.

In [37]:
antes = df_clean.shape[0]
df_clean = df_clean[(df_clean['customer_support_tickets'] >= 0) & (~df_clean['customer_support_tickets'].isin([99, 150]))]
despues = df_clean.shape[0]

print(f"Filas eliminadas: {antes - despues}")
print(f"Filas restantes: {despues}")
print(f"Rango válido: {df_clean['customer_support_tickets'].min()} - {df_clean['customer_support_tickets'].max()}")

auditar(12, 'Eliminación customer_support_tickets inválido (-1, 99, 150)', df_clean)

Filas eliminadas: 91
Filas restantes: 7414
Rango válido: 0 - 5


In [38]:
print("=== RESUMEN FINAL ===")
print(f"Filas originales: {df.shape[0]}")
print(f"Filas finales: {df_clean.shape[0]}")
print(f"Filas eliminadas en total: {df.shape[0] - df_clean.shape[0]}")
print(f"Retención: {df_clean.shape[0] / df.shape[0] * 100:.2f}%")
print(f"Columnas: {df_clean.shape[1]}")
print(f"\nNulos restantes por columna:")
print(df_clean.isnull().sum())
print(f"\nTipos de datos finales:")
print(df_clean.dtypes)

=== RESUMEN FINAL ===
Filas originales: 8160
Filas finales: 7414
Filas eliminadas en total: 746
Retención: 90.86%
Columnas: 8

Nulos restantes por columna:
user_id                     0
age                         0
subscription_plan           0
monthly_watch_time_mins     0
country                     0
favorite_genre              0
last_login_date             0
customer_support_tickets    0
dtype: int64

Tipos de datos finales:
user_id                              int64
age                                  int64
subscription_plan                      str
monthly_watch_time_mins            float64
country                                str
favorite_genre                         str
last_login_date             datetime64[us]
customer_support_tickets             int64
dtype: object


In [39]:
df_clean.to_csv('../data/processed/streaming_users_clean.csv', index=False)
print("Dataset procesado guardado en: data/processed/streaming_users_clean.csv")

Dataset procesado guardado en: data/processed/streaming_users_clean.csv


In [40]:
log_df = pd.DataFrame(log)
print(log_df.to_string(index=False))

log_df.to_csv('../logs/pipeline_log.csv', index=False)
print("\nLog guardado en: logs/pipeline_log.csv")

 Paso                                                         Descripción  Filas  Nulos  Retención (%)
    0                                            Dataset original cargado   8160    753         100.00
    1                                           Eliminación de duplicados   8034    753          98.46
    2                                     Normalización subscription_plan   8034    753          98.46
    3                                               Normalización country   8034    753          98.46
    4                                        Normalización favorite_genre   8034    753          98.46
    5                                  Imputación favorite_genre con moda   8034    513          98.46
    6                                   Eliminación nulos last_login_date   7714    186          94.53
    7 Parseo de fechas (format='mixed') y eliminación de fechas inválidas   7650    185          93.75
    8                                   Filtrado de age imposible (0-100)